In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Uniform Convergence and Generalization

The previous section put an error bar on a number computed from data. A
classifier scored on 10,000 held-out images has a true error within about 1.4
percentage points of its test error, with 95% confidence. The guarantee rests
on a hypothesis that is easy to overlook: the classifier was fixed before the
test images were drawn. Little of what we report is like that. We train many
configurations, look at their validation scores, and keep the best one. The
reported score belongs to the winner of a contest, and the winner is partly
the configuration whose luck was best.

that section met this problem and gave the
remedy of Vapnik and Chervonenkis [@Vapnik.Chervonenkis.1971]: ask for
a guarantee that holds for every candidate function at once, so that it also
holds for whichever one the learner picks. For a finite list of candidates
the guarantee widens only with the logarithm of their number. We only recall
that result
here, with one experiment that shows how much luck there is to choose from:
flip each of 1,000 fair coins 100 times and keep the coin with the most
heads. How many does it show?

The main work of the section is the infinite classes that we actually train.
For them a later idea measures a class by how well it can fit labels that are
pure coin flips, its Rademacher complexity. It explains why the capacity of
a linear model depends on its weight norm rather than on its number of
parameters, and it shows where this way of arguing stops: at models that fit
random labels, as the networks of the main text do. The last part repeats the
double-descent experiment of that section in a model small
enough that its mechanism can be computed.

In [ ]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

In [ ]:
rng = np.random.default_rng()

## Uniform Convergence

A test-set certificate and a learner differ only in the order of two events:
choosing the function and drawing the sample. This part measures how much
choosing after the sample distorts the score, and how much more data a
guarantee needs once the order is reversed.

### The Best of Many Coins

We first state the guarantee for one function in the notation used below.
A sample $S=\{(\mathbf x_1,y_1),\ldots,(\mathbf x_n,y_n)\}$ consists of $n$
independent draws from an unknown distribution. A loss $\ell$ with values in
$[0,1]$ scores a prediction, for instance the 0-1 loss, which is $1$ for a
mistake and $0$ otherwise.

The *empirical risk* of a function $f$ is its average loss on the sample,
$\hat R(f)=\frac1n\sum_{i=1}^n\ell(f(\mathbf x_i),y_i)$, and its *risk* is the
expected loss on a fresh draw, $R(f)=\mathbb{E}[\ell(f(\mathbf x),y)]$. (For
the 0-1 loss these are the empirical error $\epsilon_{\mathcal{S}}(f)$ and the
population error $\epsilon(f)$ of the main text.) If $f$ is fixed before $S$
is drawn, $\hat R(f)$ is an average of $n$ independent terms in $[0,1]$ with
mean $R(f)$, and Hoeffding's inequality
(that section) bounds
$P(|\hat R(f)-R(f)|\ge t)$ by $2e^{-2nt^2}$. Setting this to $\delta=0.05$
and solving for $t$ gives the half-width $\sqrt{\log(2/\delta)/(2n)}$, which
the main text calls the Hoeffding radius. For $n=10{,}000$ it is $0.0136$,
the 1.4 points above.

A learner reverses the order. It searches a class $\mathcal F$ of functions
and returns the $\hat f$ with the smallest $\hat R$ on the sample it saw. Each
$\hat R(f)$ fluctuates around its own $R(f)$, and the learner keeps the
function whose fluctuation was most favorable. The empirical risk
$\hat R(\hat f)$ is therefore a minimum of fluctuating quantities, and an
inequality that holds for each $f$ separately does not cover it.

The coins show by how much. Let $k=1{,}000$ fair coins each be flipped
$m=100$ times. For one coin named in advance, Hoeffding at $\delta=0.05$
allows a deviation of $\sqrt{\log 40/200}=0.136$ in the fraction of heads, so
the coin shows between 37 and 63 heads with probability at least 95% (the
exact probability is 99.3%). The best of the 1,000 coins has no such
promise. We repeat the experiment 2,000 times and record how far the best
coin strays and how often it leaves the single-coin band.

In [ ]:
k, m, repeats = 1000, 100, 2000          # coins, flips per coin, experiments
best = rng.binomial(m, 0.5, size=(repeats, k)).max(axis=1) / m
one = np.sqrt(np.log(2 / 0.05) / (2 * m))       # Hoeffding, one fixed coin
many = np.sqrt(np.log(2 * k / 0.05) / (2 * m))  # union bound over k coins
print(f'best of {k} coins: median {np.median(best):.2f}, 90% of experiments '
      f'in [{np.percentile(best, 5):.2f}, {np.percentile(best, 95):.2f}]')
print(f'95% half-width: one coin {one:.3f}, all {k} coins {many:.3f}')
print(f'experiments in which the best coin leaves the one-coin band: '
      f'{np.mean(best > 0.5 + one):.0%}')

The best coin shows 66 heads in a typical experiment and between 64 and 69 in
nine experiments out of ten. It leaves the single-coin band in 96 to 97% of
the experiments, so a guarantee that is valid for every coin named in advance
fails almost always for the coin chosen afterwards. No coin is biased. The
selected coin is off by 16 points because we looked at 1,000 fluctuations and
reported the largest.

the figure shows why. Each coin leaves the band with
probability $0.0033$ on the upper side, a rare event, but among 1,000 coins
about $3.3$ are expected to do so. The tail that one coin almost never
reaches is populated when a thousand coins are drawn.

![The expected histogram of the fraction of heads of 1,000 fair coins flipped 100 times each, on a logarithmic scale. The single-coin 95% band from Hoeffding's inequality covers 99% of the coins, yet about 3.3 coins are expected beyond its upper edge, and the best coin lies near 0.66 in a typical experiment. The band of the union bound for all 1,000 coins is wider and covers the best coin.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prob-gen-best-of-many.svg)

A validation set is the same experiment with the roles renamed: each
configuration is a coin, each example a flip. Its configurations share their
examples and differ in their true errors, but the bound below needs neither
independence nor equal means.

What would cover the chosen function? If the largest deviation over the whole
class,

$$
\sup_{f\in\mathcal F}\ \bigl|\hat R(f) - R(f)\bigr|,
$$

is at most $t$, then every function's empirical risk is within $t$ of its
risk, including the one the learner picked. Bounds on this supremum are
called *uniform convergence* bounds [@Vapnik.Chervonenkis.1971].
Choosing after seeing the data is safe if the guarantee was made for all
candidates before seeing it, because the one we pick is one of them.

### Finite Classes and the Union Bound

The simplest class is a finite list of candidates fixed in advance, such as a
grid of hyperparameter configurations. For it the main text already has the
bound, the equation, and its proof takes
two steps. Hoeffding's inequality bounds the probability that one fixed $f$
deviates by $t$ or more by $2e^{-2nt^2}$, and the union bound
(that section) over the $|\mathcal F|$ such
events gives $2|\mathcal F|e^{-2nt^2}$. Setting this to $\delta$ and solving
for $t$: for a loss with values in $[0,1]$ and an i.i.d. sample of size $n$,
with probability at least $1-\delta$,

$$
\bigl|\hat R(f) - R(f)\bigr|
 \le \sqrt{\frac{\log(2|\mathcal F|/\delta)}{2n}}
\qquad\textrm{simultaneously for every } f\in\mathcal F .
$$

Compared with one function, choosing among $|\mathcal F|$ candidates adds
$\log|\mathcal F|$ under the square root, and nothing else.

On the coins, $|\mathcal F|=1{,}000$ and $n=100$ give the half-width
$\sqrt{\log(2{,}000/0.05)/200}=0.230$, so with probability at least 95% every
coin, the best one included, shows between 27 and 73 heads. The bound is
rather conservative: the exact probability that some coin falls outside 27 to
73 heads is about 0.17%, against the 5% the bound allows. Almost all of this
factor of 30 is lost in Hoeffding's inequality for the single coin, which
leaves that range with probability $1.7\times10^{-6}$ where the inequality
allows $5\times10^{-5}$. The union bound is nearly exact for rare independent
events. Conservative or not, the bound is valid, which the single-coin band
was not.

For a validation set of 10,000 examples the main text lists the half-widths:
$0.014$ for one function and $0.030$ for a million. Solving the bound for the
sample size shows the dependence more directly:

$$
n=\frac{\log(2/\delta)+\log|\mathcal F|}{2t^2}.
$$

Each additional unit of $\log|\mathcal F|$ adds $1/(2t^2)$ examples. At a
tolerance of $t=0.01$, doubling the number of candidates costs
$\log 2/(2\cdot 10^{-4})\approx3{,}500$ more validation examples.

It is the exponential tail of Hoeffding's inequality that makes the
dependence logarithmic. With Chebyshev's inequality in its place (a
$[0,1]$-valued loss has variance at most $\tfrac14$), the same union bound
gives $t=\sqrt{|\mathcal F|/(4n\delta)}$. For 2,000 configurations on 10,000
examples at $\delta=0.05$ this is $t=1$, the whole range of the loss, where
Hoeffding gives $t=0.024$.

The proof uses one hypothesis, in its first step: the class is fixed before
the sample is drawn. A public leaderboard breaks it, because each submission
is chosen after seeing the scores of earlier ones on the same test set
[@Dwork.Feldman.Hardt.ea.2015]. The guarantee then degrades from a
deviation of order $\sqrt{\log k/n}$ for a fixed list of $k$ candidates to
one of order $\sqrt{k/n}$ for the majority vote that the main text builds
from $k$ random guessers (exercise
5).

## Rademacher Complexity

The classes we train are infinite. Every weight vector
$\mathbf w\in\mathbb{R}^d$ defines a different linear classifier, and a union
bound over infinitely many events is vacuous. The classical remedies
either replace the class by finitely many representatives (covering numbers)
or count the labelings it can produce on $n$ points (the VC dimension,
sketched in that section). The tool of this
part [@Bartlett.Mendelson.2002] asks a question a practitioner can act
on: how well can the class fit pure noise?

The two results of the section that are worth remembering are in this part:
the uniform convergence theorem the equation and the
complexity $Br/\sqrt n$ of a norm-bounded linear class,
the equation. The one hard step is the symmetrization
in the proof of the first.

### Correlation with Random Signs

If a class can fit labels that carry no information, it can also fit the
noise in real labels, and then its empirical risks say little about its
risks. To measure this we feed the class fictitious labels. Let
$\varepsilon_1,\ldots,\varepsilon_n$ be independent fair random signs, each
$+1$ or $-1$ with probability $\tfrac12$; they are called *Rademacher
variables*. For a sign vector $\boldsymbol\varepsilon$, the quantity
$\frac1n\sum_i\varepsilon_if(\mathbf x_i)$ is the correlation of $f$ with the
signs on the sample, and the best a class can do is its supremum over the
class.

**Definition.** The *empirical Rademacher complexity* of a class
$\mathcal F$ of real-valued functions on the points
$S=(\mathbf x_1,\ldots,\mathbf x_n)$ is the expected best correlation the class
achieves with random signs,

$$
\widehat{\mathfrak R}_S(\mathcal F)
 = \mathbb{E}_{\boldsymbol\varepsilon}\!\left[\,
   \sup_{f\in\mathcal F}\ \frac1n \sum_{i=1}^n \varepsilon_i\, f(\mathbf x_i)
 \right],
$$

and its *Rademacher complexity* $\mathfrak R_n(\mathcal F)=\mathbb{E}_S[\widehat{\mathfrak R}_S(\mathcal F)]$
is its average over samples of size $n$.

A class that can match any sign pattern scores high; a class that cannot
choose scores zero. The scale is set by the range of the functions. For a
class with values in $[-1,1]$ each correlation is at most $1$, and the value
$1$ is reached by the class of all functions into $[-1,1]$, which contains
$f(\mathbf x_i)=\varepsilon_i$ for every pattern (when the points are
distinct). For a class with values in $[0,1]$ the largest possible value is
$\tfrac12$: the best choice is $f(\mathbf x_i)=1$ exactly where
$\varepsilon_i=+1$, which scores the fraction of plus signs, $\tfrac12$ on
average. An unbounded class, such as the linear class below, has no such
ceiling: if it fits every sign pattern on the sample, its complexity is at
least $1$, and it can be larger.

The smallest case with something to choose is the class of the two constant
functions $\{+1,-1\}$. Its best correlation with a sign vector is
$\max(\bar\varepsilon,-\bar\varepsilon)=|\bar\varepsilon|$, where
$\bar\varepsilon$ is the average of the signs. For $n=2$ the four patterns
$(+,+),(+,-),(-,+),(-,-)$ give $|\bar\varepsilon|=1,0,0,1$, so
$\widehat{\mathfrak R}_S=\tfrac12$. For $n=4$ the sum of the signs is $\pm4$
with probability $\tfrac2{16}$, $\pm2$ with probability $\tfrac8{16}$ and $0$
otherwise, which gives $\widehat{\mathfrak R}_S=\tfrac38$. For large $n$ the
average $\bar\varepsilon$ is approximately normal with variance $1/n$, so
$\widehat{\mathfrak R}_S\approx\sqrt{2/(\pi n)}$; at $n=100$ the exact value is
$0.0796$ against $0.0798$. Even a class with only two members correlates with
noise at the rate $1/\sqrt n$, because the average of $n$ signs has that
spread. A class with one member has complexity $0$: with nothing to choose,
$\mathbb{E}[\frac1n\sum_i\varepsilon_if(\mathbf x_i)]=0$.

For a linear class the best correlation has a geometric meaning, which
the figure previews: it is the length of a signed sum
of the inputs, divided by $n$. that section proves this.

![Rademacher complexity of a linear class, for eight unit vectors in the plane and the class of unit-norm linear functions. (a) For one sign pattern, filled dots are $+1$ and hollow dots $-1$. The best correlation is attained by the weight vector $\mathbf w^*$ aligned with the signed sum $\mathbf s=\sum_i\varepsilon_i\mathbf x_i$, and equals $\|\mathbf s\|/n=0.27$. (b) The exact distribution of the best correlation over all 256 sign patterns. Its mean, the empirical Rademacher complexity, is 0.316, against the bound $Br/\sqrt n=0.354$ proved in the text.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prob-gen-rademacher.svg)

A finite class with values in $[-1,1]$ obeys *Massart's lemma*,
$\widehat{\mathfrak R}_S(\mathcal F)\le\sqrt{2\log|\mathcal F|/n}$ (exercise
2), so its complexity
grows only logarithmically with its size. Rademacher complexity measures
how freely the class can vary on the actual sample points, whatever the
number of its parameters or members.

### The Uniform Convergence Theorem

We now show that the Rademacher complexity bounds the largest deviation
between empirical risk and risk. One distinction must be made first, because
the predictions and the losses live in different classes. The predictor class
$\mathcal F$ contains the functions we fit. The functions whose averages we
compare are the losses, so define the *loss class*

$$
\mathcal G=\bigl\{(\mathbf x,y)\mapsto\ell(f(\mathbf x),y):f\in\mathcal F\bigr\},
$$

with values in $[0,1]$. For $g\in\mathcal G$ write $z=(\mathbf x,y)$,
$R(g)=\mathbb{E}[g(z)]$ and $\hat R_S(g)=\frac1n\sum_ig(z_i)$. The risk of a
predictor is the risk of its loss.

**Proposition (uniform convergence via Rademacher complexity)**
[@Bartlett.Mendelson.2002]. *Let $S=(z_1,\ldots,z_n)$ be an i.i.d.
sample and let $\mathcal G$ be a class of functions with values in $[0,1]$.
Then with probability at least $1-\delta$,*

$$
R(g) \le \hat R_S(g) + 2\,\mathfrak R_n(\mathcal G)
 + \sqrt{\frac{\log(1/\delta)}{2n}}
\qquad\textrm{simultaneously for every } g\in\mathcal G .
$$

The bound has the shape of the finite-class one: an empirical risk, a term
for the richness of the class, and a confidence term that would be there for
a single function. The richness term is twice the class's ability to fit
random signs.

**Proof.** We show
$\mathbb{E}_S\bigl[\sup_g(R(g)-\hat R_S(g))\bigr]\le2\mathfrak R_n(\mathcal G)$
in three steps; the high-probability form follows afterwards.

I. *Ghost sample.* Let $S'=(z'_1,\ldots,z'_n)$ be an independent copy of $S$,
used only inside the proof. Since $R(g)=\mathbb{E}_{S'}[\hat R_{S'}(g)]$, and a
supremum of expectations is at most the expectation of the supremum,

$$
\mathbb{E}_S\Bigl[\sup_{g\in\mathcal G}\bigl(R(g)-\hat R_S(g)\bigr)\Bigr]
 \le \mathbb{E}_{S,S'}\Bigl[\sup_{g\in\mathcal G}\ \frac1n\sum_{i=1}^n
 \bigl(g(z'_i)-g(z_i)\bigr)\Bigr].
$$

The unknown risk has been replaced by a second sample.

II. *Sign swap.* Exchanging $z_i$ with $z'_i$ leaves the joint distribution of
$(S,S')$ unchanged, because the two points are independent draws from the
same distribution. The exchange flips the sign of the $i$-th difference
$g(z'_i)-g(z_i)$ for every $g$ at once. So for any fixed signs
$\varepsilon_1,\ldots,\varepsilon_n$ the right-hand side of
the equation does not change if the $i$-th difference is
multiplied by $\varepsilon_i$. Nor does it change when the signs are drawn at
random:

$$
\mathbb{E}_{S,S'}\Bigl[\sup_{g}\ \frac1n\sum_i\bigl(g(z'_i)-g(z_i)\bigr)\Bigr]
 = \mathbb{E}_{S,S',\boldsymbol\varepsilon}\Bigl[\sup_{g}\ \frac1n\sum_i
 \varepsilon_i\bigl(g(z'_i)-g(z_i)\bigr)\Bigr].
$$

III. *Split.* The supremum of a sum is at most the sum of the suprema, so the
right-hand side is at most

$$
\mathbb{E}_{S',\boldsymbol\varepsilon}\Bigl[\sup_{g}\ \frac1n\sum_i
 \varepsilon_i\,g(z'_i)\Bigr]
 + \mathbb{E}_{S,\boldsymbol\varepsilon}\Bigl[\sup_{g}\ \frac1n\sum_i
 (-\varepsilon_i)\,g(z_i)\Bigr]
 = 2\,\mathfrak R_n(\mathcal G),
$$

because $-\boldsymbol\varepsilon$ has the same distribution as
$\boldsymbol\varepsilon$, so each term is $\mathfrak R_n(\mathcal G)$. The
split produces the factor $2$.

For the high-probability form, consider
$\Phi(S)=\sup_g(R(g)-\hat R_S(g))$. Changing one sample point changes each
$\hat R_S(g)$ by at most $1/n$, and therefore changes $\Phi$ by at most $1/n$.
*McDiarmid's inequality* [@McDiarmid.1989], the extension of Hoeffding's
inequality from sums to functions that no single input can move much, then
gives $P(\Phi-\mathbb{E}\Phi\ge t)\le e^{-2nt^2}$ (exercise
1). Setting
the right-hand side to $\delta$ and combining with the bound on
$\mathbb{E}\Phi$ gives the equation. $\blacksquare$

The intuition is that the ghost sample stands in for the population.
Swapping a real point with its ghost changes nothing, because the two are
exchangeable, and after the swaps only coin flips remain: the deviation of
the class from the truth is controlled by how well it can follow random signs.

The theorem uses the expected complexity $\mathfrak R_n$, whereas what we can
compute is $\widehat{\mathfrak R}_S$ on the sample at hand. Changing one
sample point moves $\widehat{\mathfrak R}_S(\mathcal G)$ by at most $1/n$
too, so a second use of McDiarmid's inequality gives
the equation with $\widehat{\mathfrak R}_S(\mathcal G)$
in place of $\mathfrak R_n(\mathcal G)$ and the confidence term
$3\sqrt{\log(2/\delta)/(2n)}$. The computations below rely on this form.

The theorem can be checked at its ends. For a class with one member,
$\mathfrak R_n=0$ and the equation is one-sided
Hoeffding. For a finite class, Massart's lemma turns it into
the equation with larger constants: for 2,000 configurations
on 10,000 examples the complexity term alone is $2\sqrt{2\log 2000/n}=0.078$,
against the union bound's whole half-width of $0.024$. The Rademacher route
is weaker for finite classes, but it also applies to infinite ones. For
classes of binary classifiers, counting the labelings the class can realize
on $n$ points bounds $\mathfrak R_n$ by a constant times
$\sqrt{d_{\mathrm{VC}}\log n/n}$ [@boucheron2005theory]. Here
$d_{\mathrm{VC}}$ is the VC dimension of the main text, the largest number of
points on which the class realizes every labeling.

At the other end, a loss class with values in $[0,1]$ has
$\mathfrak R_n(\mathcal G)\le\tfrac12$. When equality holds, the complexity
term is $1$ and the bound allows any risk whatever. For classifiers with the
0-1 loss, the two complexities stand in a fixed ratio. Let the predictors
take values in $\{-1,+1\}$ and use the 0-1
loss, $g(\mathbf x,y)=(1-yf(\mathbf x))/2$. Then
$\sum_i\varepsilon_ig(z_i)=\tfrac12\sum_i\varepsilon_i+\tfrac12\sum_i(-\varepsilon_iy_i)f(\mathbf x_i)$.
The first sum has mean zero and does not depend on $f$, and for fixed labels
$-\varepsilon_iy_i$ is again a fair random sign, so

$$
\widehat{\mathfrak R}_S(\mathcal G)=\tfrac12\,\widehat{\mathfrak R}_S(\mathcal F).
$$

Such a predictor class, if it fits every sign pattern on the sample, has
$\widehat{\mathfrak R}_S(\mathcal F)=1$. Twice the loss complexity is then
$1$, and the theorem promises only an error rate below 100%.

### The Linear Class

The theorem is useful only if the complexity can be computed for a class we
care about. For linear functions with a bounded weight norm the computation
takes four lines, and its answer does not involve the dimension.

**Proposition (Rademacher complexity of a norm-bounded linear class).** *Let
$\mathcal F=\{\mathbf x\mapsto\langle\mathbf w,\mathbf x\rangle :
\|\mathbf w\|_2\le B\}$ and suppose the sample satisfies
$\|\mathbf x_i\|_2\le r$ for all $i$. Then*

$$
\widehat{\mathfrak R}_S(\mathcal F)
 = \frac Bn\,\mathbb{E}_{\boldsymbol\varepsilon}\Bigl\|\sum_{i=1}^n\varepsilon_i\mathbf x_i\Bigr\|
 \le \frac{B\,r}{\sqrt n}.
$$

**Proof.** The supremum is explicit. By linearity and Cauchy--Schwarz, with
equality at $\mathbf w$ of length $B$ aligned with the signed sum,

$$
\sup_{\|\mathbf w\|\le B}\ \frac1n\sum_{i=1}^n
 \varepsilon_i \langle\mathbf w,\mathbf x_i\rangle
 = \sup_{\|\mathbf w\|\le B}\ \frac1n
   \Bigl\langle\mathbf w,\ \sum_i \varepsilon_i\mathbf x_i\Bigr\rangle
 = \frac Bn\,\Bigl\|\sum_{i=1}^n \varepsilon_i\mathbf x_i\Bigr\| .
$$

Taking the expectation gives the equality in the proposition. For the bound,
the square root is concave, so $\mathbb{E}\|\cdot\|\le\sqrt{\mathbb{E}\|\cdot\|^2}$,
and expanding the square,

$$
\mathbb{E}_{\boldsymbol\varepsilon}\Bigl\|\sum_i\varepsilon_i\mathbf x_i\Bigr\|
 \le \sqrt{\sum_{i,j} \mathbb{E}[\varepsilon_i\varepsilon_j]\,
   \langle\mathbf x_i,\mathbf x_j\rangle}
 = \sqrt{\sum_i \|\mathbf x_i\|^2}
 \le r\sqrt n .
$$

The cross terms vanish because independent signs have
$\mathbb{E}[\varepsilon_i\varepsilon_j]=0$ for $i\ne j$. Multiplying by $B/n$
gives $Br/\sqrt n$. $\blacksquare$

Panel (a) of the figure draws the first step of this
proof: the best weight vector points along the signed sum.

Only one step of the proof loses anything. The identity
$\mathbb{E}\|\sum_i\varepsilon_i\mathbf x_i\|^2=\sum_i\|\mathbf x_i\|^2$ is
exact, so the slack is all in the step from the mean length of the signed sum
to its root-mean-square length. That step is tight when the length varies
little from one sign pattern to the next. For the eight points of the figure
the slack is 11% ($0.316$ against $0.354$). In the plane, a signed sum of many
unit vectors in spread-out directions is approximately a two-dimensional
Gaussian vector, whose length has mean $\sqrt\pi/2\approx0.89$ times its
root-mean-square, so the slack stays near 11% however many points there are.
In twenty dimensions the length of the signed sum concentrates, by the thin
shell of that section, and the slack falls to about 1%.

The dimension $d$ does not appear in the equation: it
matters only through its effect on $B$ and $r$. In this bound the capacity of
a linear model is set by the product of its weight norm and its input radius,
whether it acts on ten features or on a million. Weight decay
(that section) keeps $\|\mathbf w\|$ small, and so keeps the
class that contains the fitted model small. Nor is this confined to linear
models: a two-layer ReLU network with output weights of $\ell_1$ norm at most
$B_1$ and hidden weight vectors of norm at most $B_2$ has complexity at most
$2B_1B_2r/\sqrt n$, whatever its width (exercise
10).

A reader may object that the proposition needs $B$ fixed before the data,
while the norm of a trained weight vector is known only afterwards. The
objection is right, and a union bound over radii answers it. Apply the bound
to each radius $B_j=2^j$ with failure probability $\delta_j=\delta/(j(j+1))$,
which sum to $\delta$, and use the smallest radius that contains the fitted
vector. For $\|\mathbf w\|>1$ this loses at most a factor $2$ in the
complexity term and adds a term of order $\sqrt{\log\log\|\mathbf w\|/n}$
(exercise
3).

The theorem needs the complexity of the loss class, and the proposition
computes that of the predictor class. A Lipschitz loss connects them.

**Lemma (contraction)** [@Boucheron.Lugosi.Massart.2013]. *Let
$\phi_1,\ldots,\phi_n:\mathbb{R}\to\mathbb{R}$ each be $L$-Lipschitz. Then for
any class $\mathcal F$ and points $\mathbf x_1,\ldots,\mathbf x_n$,*

$$
\mathbb{E}_{\boldsymbol\varepsilon}\Bigl[\sup_{f\in\mathcal F}\sum_{i=1}^n
 \varepsilon_i\,\phi_i\bigl(f(\mathbf x_i)\bigr)\Bigr]
 \le L\,\mathbb{E}_{\boldsymbol\varepsilon}\Bigl[\sup_{f\in\mathcal F}
 \sum_{i=1}^n\varepsilon_i\,f(\mathbf x_i)\Bigr].
$$

Composing with a function that stretches distances by at most $L$ multiplies
the complexity by at most $L$. Each $\phi_i$ may differ, which is what a loss
needs: at the point $i$ the loss is the function $u\mapsto\ell(u,y_i)$ of the
prediction, with the label $y_i$ fixed. No centering is required, since adding
a constant $c$ to $\phi_i$ adds $c\,\varepsilon_i$, whose expectation is zero.
We state the lemma without proof.

The theorem also needs losses in $[0,1]$. The hinge and logistic losses are
not bounded by $1$: on the class they reach about $1+Br$, so they must be
divided by that maximum before the theorem applies, and the result is a
bound on the rescaled loss. A cleaner route is a loss that is bounded from
the start, the *ramp loss* on the margin $u=yf(\mathbf x)$,

$$
\phi_\gamma(u)=\min\bigl\{1,\max\{0,\,1-u/\gamma\}\bigr\},
$$

which is $1$ for a mistake, $0$ for a correct prediction with margin at least
$\gamma$, and linear in between. It lies above the 0-1 loss and is
$(1/\gamma)$-Lipschitz.

Write $\hat R_\gamma(\mathbf w)$ for the average of the ramp loss
$\frac1n\sum_i\phi_\gamma(y_i\langle\mathbf w,\mathbf x_i\rangle)$ on the
sample. With $\|\mathbf x\|\le r$ for every input, the contraction lemma and
the linear-class proposition give a loss-class complexity of at most
$Br/(\gamma\sqrt n)$, and the theorem gives, with probability at least
$1-\delta$, for every $\mathbf w$ with $\|\mathbf w\|\le B$,

$$
P\bigl(y\langle\mathbf w,\mathbf x\rangle\le0\bigr)
 \le \hat R_\gamma(\mathbf w) + \frac{2Br}{\gamma\sqrt n}
 + \sqrt{\frac{\log(1/\delta)}{2n}} .
$$

The left side is the true error rate of the linear classifier; the right side
is its training ramp loss plus two terms we can evaluate. As an illustration
(the ratio is chosen, not measured), take $Br/\gamma=10$, $n=50{,}000$ and
$\delta=0.05$. The complexity term is $0.089$ and the confidence term
$0.005$, together $0.095$. With $B=10\gamma/r$ fixed in advance, a linear
classifier in the class that separates its training data with margin $\gamma$
has, with probability at least 95%, a true error below $9.5\%$. Fitting random labels with a margin
forces $Br/\gamma$ up. Once it passes about $10.6$ the two terms exceed
$0.1$, and near $Br/\gamma\approx110$ the complexity term alone reaches $1$.

### Rademacher Complexity in Numbers

The proposition gives the complexity of the linear class as an expected
length, and the expectation can be estimated by averaging over random sign
vectors. We do this for 50 unit-norm points in $d=2$, $20$ and $1{,}000$
dimensions. The slack analysis above predicts an estimate about 11% below
the bound $Br/\sqrt n=1/\sqrt{50}$ in the plane and about 1% below it in
twenty dimensions.

In [ ]:
n, B, num_signs = 50, 1.0, 2000
signs = rng.choice([-1.0, 1.0], size=(num_signs, n))  # rows: sign vectors
for d in (2, 20, 1000):
    X = rng.standard_normal((n, d))
    X /= np.linalg.norm(X, axis=1, keepdims=True)       # ||x_i|| = r = 1
    sup_corr = B * np.linalg.norm(signs @ X, axis=1) / n  # closed-form sup
    print(f'd = {d:4d}: Monte-Carlo R_S = {sup_corr.mean():.3f}, '
          f'bound B r / sqrt(n) = {B / np.sqrt(n):.3f}')

The estimate is about $0.125$ in the plane and $0.139$ to $0.141$ in 20 and
1,000 dimensions, against the bound $0.141$, as predicted. Going from 20 to
1,000 dimensions leaves the complexity unchanged.

The other end of the scale is a class rich enough to *interpolate*, that is,
to reproduce any labels exactly on the sample. Least squares on $p=2n$
Gaussian features, fit with the pseudoinverse of
that section, returns for every sign vector the
interpolant $\mathbf w=\boldsymbol\Phi^+\boldsymbol\varepsilon$ of smallest
norm, where $\boldsymbol\Phi\in\mathbb{R}^{n\times p}$ holds the features of
the points as rows. Its correlation with the signs is $1$ for every pattern,
by construction. What the proposition tells us is how large a ball of
weights this takes: a class that interpolates every sign pattern has
$\widehat{\mathfrak R}_S\ge1$, the proposition gives
$\widehat{\mathfrak R}_S\le Br/\sqrt n$, hence any ball that contains all the
interpolants has $Br/\sqrt n\ge1$. How close to this minimum do actual fits
come? The cell fits 2,000 sign vectors and prints their norms (the predicted
mean $\sqrt{n/(p-n-1)}$ is derived in exercise
4) and the
value of $Br/\sqrt n$ for the smallest ball that contains them.

In [ ]:
p = 2 * n                                     # more features than points
Phi = rng.standard_normal((n, p))
W = np.linalg.pinv(Phi) @ signs.T             # min-norm fit of each sign vector
w_norm = np.linalg.norm(W, axis=0)
r = np.linalg.norm(Phi, axis=1).max()         # input radius: largest row norm
row_norm = np.linalg.norm(Phi, axis=1).mean()
print(f'mean ||w|| = {w_norm.mean():.2f} (predicted '
      f'{np.sqrt(n / (p - n - 1)):.2f}), largest {w_norm.max():.2f}')
print(f'typical fit: mean ||w|| * mean row norm / sqrt(n) = '
      f'{w_norm.mean() * row_norm / np.sqrt(n):.2f}')
print(f'r = {r:.1f}, B r / sqrt(n) with B = largest ||w||: '
      f'{w_norm.max() * r / np.sqrt(n):.1f}')

The fits have norm about $1$ and the rows of $\boldsymbol\Phi$ length about
$\sqrt p=10$, so a typical fit sits at
$\|\mathbf w\|r/\sqrt n\approx1.4$. The largest of the 2,000 norms is 1.3 to
1.8 and the longest row 11 to 12.5, which puts the ball around all the fits
at 2 to 3. Interpolating noise therefore takes a norm only one and a half
to three times the minimum that the proposition allows: the bound
$Br/\sqrt n$ is nearly tight for this class, even at a complexity where it
has stopped being useful.

The same arithmetic applies to networks. Take the class of all classifiers
that an architecture can represent. If training on labels
$\boldsymbol\varepsilon$ returns a member that agrees with them at nearly
every point, the supremum in the equation is close to $1$ for
that $\boldsymbol\varepsilon$, and if this happens for typical sign vectors,
so is the complexity. In that section an MLP fit random
labels of its training images exactly, as the image classifiers of
@zhang2021understanding did. (These experiments tried a few
labelings with ten classes, so they are evidence, not a proof.) Twice the
complexity of the loss class is then about $1$, and
the equation allows any error rate. Yet the same network
generalizes on real labels. What decides its test error is therefore
invisible to a bound over the whole class: the data, the optimizer, and the
solution it reached. The next part examines a model in which the solution
can be computed in closed form.

## Interpolation and Double Descent

that section swept the width $h$ of a network whose hidden
layer stayed frozen at its random initialization. The test error peaked at
$h=n$, where an exact fit first became possible and the smallest singular
value of the hidden representation was tiny, and fell again beyond it. Here
we repeat the construction on synthetic data, where every quantity in the
mechanism can be computed, and add two things to the main text's account: a
formula for the fit in terms of singular values, and a split of the test
error into the part due to the target and the part due to label noise. The
features are $\boldsymbol\phi(\mathbf x)=\mathrm{ReLU}(\mathbf V\mathbf x)$
with a frozen random matrix $\mathbf V$, a *random-features* model
[@Rahimi.Recht.2007], and only the linear head $\mathbf w$ is fit.
This part can be skipped on a first reading.

### The Minimum-Norm Interpolant

To see why the threshold is special we need the solution in closed form. With
$p$ features and $n$ points, the feature matrix
$\boldsymbol\Phi\in\mathbb{R}^{n\times p}$ has the feature vectors of the
training points as rows, and we fit $\mathbf w=\boldsymbol\Phi^+\mathbf y$:
least squares below $p=n$, the interpolant of smallest norm above it.

The singular value decomposition shows what happens below, at and above the
threshold. Let
$\boldsymbol\Phi$ have full rank $q=\min(n,p)$ and singular value
decomposition $\boldsymbol\Phi=\sum_{i=1}^q\sigma_i\mathbf u_i\mathbf v_i^\top$,
with $\sigma_1\ge\cdots\ge\sigma_q>0$. Then

$$
\mathbf w=\boldsymbol\Phi^+\mathbf y=\sum_{i=1}^{q}
 \frac{\langle\mathbf u_i,\mathbf y\rangle}{\sigma_i}\,\mathbf v_i .
$$

Every component of $\mathbf y$ along a left singular vector is divided by its
singular value, so the smallest singular value $\sigma_{\min}=\sigma_q$
amplifies most. Whatever part of $\mathbf y$ lies along $\mathbf u_q$ enters
the solution with the factor $1/\sigma_{\min}$. This holds for label noise
and equally for the part of the target that the features represent badly.

For $p<n$ the fit $\boldsymbol\Phi\mathbf w$ is the projection of $\mathbf y$
onto the $p$ directions $\mathbf u_i$, and the rest of $\mathbf y$ is left as
residual. At $p=n$ there are $n$ directions, the fit is exact, and nothing is
left over to absorb the badly represented part. A random square ReLU-feature
matrix is almost always invertible, but rarely *well* invertible: its
smallest singular value is small, and the condition-number analysis of
that section says that the solution then moves far
in the direction $\mathbf v_q$. Adding features beyond $n$ gives the matrix
more room, and its smallest singular value grows again.

Past the threshold the norm of the solution can only fall as features are
added, provided the features are nested (the model with $p+1$ features uses
the features of the model with $p$, plus one more). This is the padding
argument of that section: give the new feature zero
weight, and the minimum-norm interpolant with $p$ features becomes an
interpolant with $p+1$ features of the same norm. The argument needs exact
interpolation. Below $p=n$ the least-squares solution is not an interpolant,
and its norm grows with $p$: in the sweep below it rises from about $0.5$ at
$p=2$ to nearly $2$ at $p=15$.

The linear-class bound suggests a quantity to track along the sweep: the
*capacity product*
$\|\mathbf w\|\cdot\max_i\|\boldsymbol\phi(\mathbf x_i)\|/\sqrt n$, which is
$Br/\sqrt n$ for the smallest ball that contains the fitted head. It is
borrowed from classification. The sweep is a regression with squared loss on
unbounded targets, which no theorem of this section covers, so the product
is a diagnostic and not a bound. The feature radius grows like $\sqrt p$
while the norm of the solution falls past the threshold, so the product
could go either way.

### A Double-Descent Sweep

The data are forty training points from a linear teacher
$y=\langle\boldsymbol\beta,\mathbf x\rangle$ in fifteen dimensions, with
$\|\boldsymbol\beta\|=1$ and label noise of standard deviation $0.1$. The
features are ReLU random features with nested columns, and the head is fit
by `pinv`: least squares below the threshold, the minimum-norm
interpolant above it. We sweep $p$ from $2$ to $400$ through $p=n=40$, with
the neighbours $p=39$ and $41$ included so that the peak is resolved. For
every fit we record the test error, the training error, $\|\mathbf w\|$,
$\sigma_{\min}$ and the capacity product, and also the two parts of the test
error that that section separates: the clean part,
the error of the same fit made to noise-free labels, and the noise part, the
extra error that the label noise adds. Medians over 20 independent runs keep
the curve readable.

In [ ]:
n, n_test, d_in, p_max, trials, noise = 40, 500, 15, 400, 20, 0.1
p_grid = np.array([2, 5, 10, 15, 20, 25, 30, 35, 38, 39, 40, 41, 42, 45, 50,
                   60, 80, 120, 200, 400])
keys = ('test', 'train', 'norm', 's_min', 'capacity', 'clean', 'noise',
        'ratio_q', 'ratio')
stats = np.zeros((len(keys), trials, len(p_grid)))
ols = np.zeros(trials)                               # least squares on x itself
for tr in range(trials):
    beta = rng.standard_normal(d_in)
    beta /= np.linalg.norm(beta)                     # teacher: y = <beta, x>
    X, X_te = (rng.standard_normal((size, d_in)) for size in (n, n_test))
    y_clean, y_te = X @ beta, X_te @ beta
    y = y_clean + noise * rng.standard_normal(n)     # noisy training labels
    ols[tr] = np.mean((X_te @ np.linalg.lstsq(X, y, rcond=None)[0] - y_te) ** 2)
    V = rng.standard_normal((p_max, d_in)) / np.sqrt(d_in)
    Phi, Phi_te = np.maximum(X @ V.T, 0), np.maximum(X_te @ V.T, 0)
    for j, p in enumerate(p_grid):                   # nested feature prefixes
        F, F_te = Phi[:, :p], Phi_te[:, :p]
        P = np.linalg.pinv(F)                        # least squares / min norm
        w = P @ y
        r = np.linalg.norm(F, axis=1).max()          # feature radius
        _, S, Vt = np.linalg.svd(F, full_matrices=False)
        ratio = ((np.linalg.norm(F_te @ Vt.T, axis=0) ** 2 / n_test)
                 / (S ** 2 / n))                     # test/train feature power
        stats[:, tr, j] = (
            np.mean((F_te @ w - y_te) ** 2),                 # test MSE
            np.mean((F @ w - y) ** 2),                       # training MSE
            np.linalg.norm(w),                               # ||w||
            S.min(),                                         # sigma_min
            np.linalg.norm(w) * r / np.sqrt(n),              # capacity
            np.mean((F_te @ (P @ y_clean) - y_te) ** 2),     # clean part
            noise ** 2 * np.linalg.norm(F_te @ P) ** 2 / n_test,  # noise part
            ratio[-1], ratio.mean())                 # along v_q, average
med = dict(zip(keys, np.median(stats, axis=1)))

The plot shows the medians of the test error, the solution norm and the
capacity product on logarithmic axes.

In [ ]:
d2l.plot([p_grid] * 3 + [[n, n]],
         [med['test'], med['norm'], med['capacity'], [1e-2, 1e2]],
         'number of random features p', 'median over 20 runs',
         legend=['test MSE', '||w||', 'capacity', 'p = n'], xscale='log',
         yscale='log', fmts=('-', 'm--', 'g-.', 'k:'), figsize=(6, 3.5))

The table gives the numbers at seven feature counts. Its last two columns
belong to that section, and its last line is the
classical model that fits the raw inputs.

In [ ]:
print('   p ' + ' '.join(f'{key:>8}' for key in keys))
for p in (15, 38, 40, 42, 50, 80, 400):
    row = [med[key][p_grid == p][0] for key in keys]
    print(f'{p:4d} ' + ' '.join(f'{v:8.2g}' for v in row))
print(f'least squares on the {d_in} raw inputs: test MSE {np.median(ols):.4f}')

The table is best read in three stretches: below the threshold, at it, and
beyond it.

*Below the threshold* the model is a classical one, with fewer parameters
than points. Its test error first falls, from about $1$ at $p=2$ to roughly
$0.6$ to $0.8$ near $p=15$, and then rises again as $p$ approaches $40$.
This is the U-curve that the classical account leads us to expect. Two
things must be said about this regime before anything is compared with it.
First, it is weak. The targets have variance $1$, so a test error of $1$ is
what predicting zero achieves, and the best fit here removes only a quarter
to a third of it. Second, a far better classical model exists. The teacher
is linear in the fifteen raw inputs, and least squares on those inputs has
an expected test error of $0.01\cdot15/(40-15-1)\approx0.006$, the noise
variance times $d/(n-d-1)$ for $d=15$. The last line of the table confirms
it. The random features were handicapped by construction: they have to
assemble a linear function out of ReLU ridges, and forty points are too few
for that.

*At the threshold* three columns move together. The training error drops
from a few times $10^{-3}$ at $p=38$ to about $10^{-28}$, which is exact
interpolation up to floating point. The smallest singular value falls to
about $0.01$ to $0.04$, several times below its values at $p=38$ and $p=42$
on either side. And since the equation divides by it, the
solution is inflated: the norm jumps to between about $6$ and $20$, and the
test error to somewhere between $5$ and $50$, one to two orders of magnitude
above the minimum of the first stretch. These ranges are wide because the
peak is heavy-tailed, as it is for a square Gaussian matrix, where
$\mathbb{E}[1/\sigma_{\min}^2]$ is infinite. Even a median over 20 trials
moves from one run of the cell to the next.

*Beyond the threshold* everything that rose comes down again. The smallest
singular value recovers, to about $2.8$ at $p=400$. The norm falls steadily,
as nesting requires (in every trial, not only in the medians), down to about
$0.4$. The test error follows: about $0.2$ at $p=80$, and $0.04$ to $0.07$
at $p=400$. One column has nothing more to tell us. The training error is
zero up to floating point everywhere past the threshold, so no criterion
based on it can separate $p=42$ from $p=400$, whose test errors differ by
well over an order of magnitude. The norm, the smallest singular value and the
capacity product all can.

What, then, has the sweep shown? Less than its plot suggests. The widest
model beats every narrower model of its own family, the best one below the
threshold included, by roughly a factor of ten. It still loses to least
squares on the raw inputs by about another factor of ten. Within one nested
family of random features, more features are better, even far past the
point where the model fits its noise. That is a real and surprising fact.
Whether interpolation can beat a well-specified small model is a different
question, and in this experiment the answer is no.

The capacity product peaks at the threshold with values of about 8 to 25,
falls past it, and reaches about $1.2$ to $1.3$ at $p=400$. It follows the
shape of the test curve, but it is no guarantee: no theorem here covers this
loss, the product changes when the targets are rescaled, and its $B$ was
read off the fit rather than fixed in advance.

The peak is not caused by the label noise. Rerunning the sweep with
`noise = 0` leaves a peak of the same size, and the table shows why. At
$p=40$ the clean part of the test error is larger than the noise part,
usually several times larger. The linear teacher is not a combination of
finitely many ReLU features, and the part of the target that the features
represent badly is what $\sigma_{\min}$ divides. With a target that is an
exact combination of the first ten features and no noise, there is no peak
at all (exercise 8).

### Benign Overfitting

One puzzle remains. The $p=400$ model reproduces every noisy training label
exactly, yet it is the best of the sweep. Why does fitting the noise cost so
little there?

The test error separates into two parts we can compute. Write the training
labels as $\mathbf y=\mathbf y_0+\boldsymbol\xi$, with $\mathbf y_0$ the
noise-free labels and $\boldsymbol\xi$ independent noise of variance
$\sigma_\xi^2=0.01$ per label. Then
$\mathbf w=\boldsymbol\Phi^+\mathbf y_0+\boldsymbol\Phi^+\boldsymbol\xi$, and the
predictions at the test points are
$\boldsymbol\Phi_{\mathrm{te}}\boldsymbol\Phi^+\mathbf y_0+\boldsymbol\Phi_{\mathrm{te}}\boldsymbol\Phi^+\boldsymbol\xi$.
In the squared test error, the cross term between these two parts has mean
zero because $\boldsymbol\xi$ does. Averaged over
the noise, the test error over $n_{\mathrm{te}}$ test points is

$$
\mathbb{E}_{\boldsymbol\xi}\bigl[\textrm{test error}\bigr]
 = \underbrace{\frac{1}{n_{\mathrm{te}}}\bigl\|\boldsymbol\Phi_{\mathrm{te}}\boldsymbol\Phi^+\mathbf y_0-\mathbf y_{\mathrm{te}}\bigr\|^2}_{\textrm{clean part}}
 + \underbrace{\frac{\sigma_\xi^2}{n_{\mathrm{te}}}\bigl\|\boldsymbol\Phi_{\mathrm{te}}\boldsymbol\Phi^+\bigr\|_F^2}_{\textrm{noise part}} .
$$

The last two columns of the table print the two parts. At $p=40$ the noise
part is about $1$ to $4$, a hundred times or more the noise variance $0.01$,
and the clean part is larger still. At $p=42$ the noise part is a few
tenths, at $p=80$ about $0.02$, and at $p=400$ just under $0.01$. The clean
part falls from roughly 4 to 20 at the peak to about $0.05$ at
$p=400$, so most of the gain past the threshold comes from a better
approximation of the target.

Why does the division by small singular values, which produced the peak, do
so little harm at $p=400$? By the equation a noise
component along $\mathbf u_i$ enters the weights with the factor $1/\sigma_i$
along $\mathbf v_i$, and it moves a test prediction by that weight times the
component of the test feature vector along $\mathbf v_i$. Since
$\|\boldsymbol\Phi_{\mathrm{te}}\boldsymbol\Phi^+\|_F^2=\sum_i\|\boldsymbol\Phi_{\mathrm{te}}\mathbf v_i\|^2/\sigma_i^2$
and $\sigma_i^2=\|\boldsymbol\Phi\mathbf v_i\|^2$, the noise part is, for
$p\ge n$,

$$
\sigma_\xi^2\cdot\frac1n\sum_{i=1}^{n}
 \frac{\|\boldsymbol\Phi_{\mathrm{te}}\mathbf v_i\|^2/n_{\mathrm{te}}}
      {\|\boldsymbol\Phi\mathbf v_i\|^2/n} ,
$$

the noise variance times the average over directions of the ratio of the
mean squared test feature to the mean squared training feature along
$\mathbf v_i$. A small $\sigma_i$ hurts only if the test features are not
small in the same direction. At $p=n$ the smallest singular value is small
by an accident of the forty training points: along $\mathbf v_q$ the test
features are typically thousands of times larger in mean square than the
training features. At $p=400$ a small $\sigma_i$ marks a direction in which
the features vary little for training and test points alike. The table's
last two columns show it: the ratio along $\mathbf v_q$ (`ratio_q`) is about
$2$, and the average over all directions (`ratio`) is just below $1$, so the
small singular value in the denominator is cancelled by a small test feature
in the numerator. Overfitting is called *benign* in this sense. Benign means
bounded here, not vanishing: at $p=400$ the predictions still carry about as
much noise as a single training label.

@Bartlett.Long.Lugosi.Tsigler.2020 make this precise for linear
regression, through two effective ranks of the covariance of the inputs, and
exercise 9 measures
them. In deep networks the features are learned and change the spectrum
during training, and a theory that predicts a given network's test error from
first principles is still open. Every bound in this section covers the
worst function in a class fixed before the data, while the function we use
is the one an algorithm reached. A guarantee that depends on the algorithm
and the data, rather than on the class alone, is the open question this
section leaves.

## Summary

A guarantee for one function fixed before the data does not cover the
function chosen because it looked best on the data. Uniform convergence
repairs this by bounding the deviation of every function in the class at
once. For a finite class, Hoeffding's inequality and the union bound do so
with $\log|\mathcal F|/(2t^2)$ additional examples at tolerance
$t$. For infinite classes, symmetrization with a ghost sample turns the
unknown risk into a correlation with random signs, and the bound adds twice
the Rademacher complexity of the loss class. For the linear class with
weight norm $B$ and inputs of norm $r$ this complexity is at most
$Br/\sqrt n$, whatever the dimension. The argument stops at classes that fit
every sign pattern, such as a network that fits random labels: for them the
bound allows any error rate. In the random-features sweep the test error
peaks where $\sigma_{\min}$ is smallest and falls as the minimum-norm
solution shrinks, and these quantities belong to the solution rather than to
the class.

## Exercises

1. **McDiarmid's inequality.** ● The inequality of @McDiarmid.1989
   extends Hoeffding's from sums to functions with *bounded differences*.
   Suppose that changing only the $i$-th argument of $g(x_1,\ldots,x_n)$
   moves its value by at most $c_i$, and that $X_1,\ldots,X_n$ are
   independent. Then

   $$P\bigl(g(X_1,\ldots,X_n)-\mathbb{E}[g(X_1,\ldots,X_n)]\ge t\bigr)\le\exp\left(-\frac{2t^2}{\sum_{i=1}^n c_i^2}\right),$$

   and the same bound holds for the lower tail. Take this statement on
   faith; its proof uses martingales.
    1. Recover Hoeffding's inequality for an average of independent terms
       $X_i\in[a_i,b_i]$ from it.
    1. Let $\Phi(S)=\sup_{g\in\mathcal G}\bigl(R(g)-\hat R_S(g)\bigr)$ for a
       loss class $\mathcal G$ with values in $[0,1]$. Show that $\Phi$ has
       bounded differences $c_i=1/n$. Combine McDiarmid's inequality with the
       symmetrization bound $\mathbb{E}_S[\Phi(S)]\le 2\mathfrak R_n(\mathcal G)$
       of the text and derive the equation.
    1. For data in $[0,1]$, the sample mean has bounded differences $1/n$.
       Construct a sample of odd size $n$ in $[0,1]$ in which changing one
       point moves the sample median by $1$. What does this imply for using
       McDiarmid's inequality to put an error bar on the bootstrap estimate
       of the median in that section?

1. **Massart's lemma and $\ell_1$ balls.** ● Massart's lemma for finite
   classes and the Rademacher complexity of $\ell_1$-bounded linear classes
   rest on one bound for the expected maximum of sub-Gaussian variables.
    1. Let $Z_1,\ldots,Z_N$ be centered and sub-Gaussian with variance proxy
       $\sigma^2$, that is,
       $\mathbb{E}[e^{\lambda Z_j}]\le e^{\lambda^2\sigma^2/2}$ for all
       $\lambda$, but not necessarily independent. Show that
       $\mathbb{E}[\max_j Z_j]\le\sigma\sqrt{2\log N}$ by bounding
       $e^{\lambda \mathbb{E}[\max_j Z_j]}$ by $\sum_j \mathbb{E}[e^{\lambda Z_j}]$ and
       optimizing over $\lambda>0$. Deduce Massart's lemma,
       $\widehat{\mathfrak R}_S(\mathcal F)\le\sqrt{2\log|\mathcal F|/n}$,
       for a finite class with values in $[-1,1]$.
    1. Let $\mathcal F=\{\mathbf x\mapsto\langle\mathbf w,\mathbf x\rangle:
       \|\mathbf w\|_1\le B\}$ and $\|\mathbf x_i\|_\infty\le r$ for all $i$.
       Follow the proof of the equation, with the
       duality $\sup_{\|\mathbf w\|_1\le B}\langle\mathbf w,\mathbf
       v\rangle=B\|\mathbf v\|_\infty$ in place of Cauchy--Schwarz, and show
       that $\widehat{\mathfrak R}_S(\mathcal F)\le Br\sqrt{2\log(2d)/n}$.
       Why does the dimension $d$ now enter, and why only logarithmically?
    1. Let the inputs lie in $\{-1,+1\}^d$ and let the target weight vector
       have $k$ nonzero entries, each equal to $\pm1$. Compare the bound
       of (b) with the equation, each with the smallest
       radius $B$ that contains the target. For which $k$ is the $\ell_1$
       bound smaller when $d=10^4$?

1. **Margin bound.** ● Let $\mathcal F=\{\mathbf x\mapsto\langle\mathbf
   w,\mathbf x\rangle:\|\mathbf w\|_2\le B\}$ with $\|\mathbf x\|_2\le r$,
   let the labels be $y\in\{-1,+1\}$, and let the ramp loss
   $\phi_\gamma(u)=\min\{1,\max\{0,1-u/\gamma\}\}$ with a fixed $\gamma>0$
   act on the margin $u=yf(\mathbf x)$.
    1. Show that $\phi_\gamma(yf(\mathbf x))$ is at least the 0-1 loss
       $\mathbf 1\{yf(\mathbf x)\le 0\}$ and that $\phi_\gamma$ is
       $(1/\gamma)$-Lipschitz. Show that multiplying each prediction
       $f(\mathbf x_i)$ by a fixed label $y_i$ leaves
       $\widehat{\mathfrak R}_S(\mathcal F)$ unchanged.
    1. Combine (a), the contraction lemma the equation,
       the equation, and
       the equation into the equation.
       For $n=50{,}000$ and $\delta=0.05$, find the largest ratio $Br/\gamma$
       for which the complexity and confidence terms together stay below
       $0.1$.
    1. The bound holds for a radius $B$ fixed before the data are drawn,
       whereas the norm of a trained weight vector is known only afterwards.
       Apply the bound with radius $B_j=2^j$ and confidence
       $\delta_j=\delta/(j(j+1))$ for every $j\ge1$, and take a union bound
       over $j$. State the bound that holds for a weight vector with
       $\|\mathbf w\|>1$ chosen after seeing the data. Show that the freedom
       to choose the radius costs at most a factor $2$ in the complexity term
       and an additional term of order $\sqrt{\log\log\|\mathbf w\|/n}$.

1. [code] **Fitting random signs.** ● The second cell of
   that section fits $2{,}000$ sign vectors
   $\boldsymbol\varepsilon\in\{-1,+1\}^n$ with
   $\mathbf w=\boldsymbol\Phi^+\boldsymbol\varepsilon$ on $p=2n$ standard
   Gaussian features and measures the norms of the fits.
    1. Let $p<n$ and let $\boldsymbol\Phi$ have full column rank. Show that
       the correlation $\frac1n\sum_i\varepsilon_i(\boldsymbol\Phi\mathbf w)_i$
       has expectation $p/n$ over the signs. The matrix
       $\boldsymbol\Phi\boldsymbol\Phi^+$ is a projection.
    1. Let $p>n$, let $r$ be the largest row norm of $\boldsymbol\Phi$, and
       let $B$ be large enough that the class $\{\|\mathbf w\|\le B\}$
       contains an interpolant of every sign vector. Explain why
       $Br/\sqrt n$ cannot then be smaller than $1$. For a standard Gaussian
       $\boldsymbol\Phi$ with $p>n+1$,
       $\mathbb{E}[(\boldsymbol\Phi\boldsymbol\Phi^\top)^{-1}]=\mathbf I_n/(p-n-1)$.
       Use this identity to predict the typical $\|\mathbf w\|$, and the
       typical value of $\|\mathbf w\|$ times a typical row norm of
       $\boldsymbol\Phi$ divided by $\sqrt n$, as functions of $p$ and
       $n$.
    1. Rerun the cell with $n=50$ for $p\in\{10,25,45,55,100,400,1000\}$
       (for $p<n$ print the correlation instead of the norm) and compare
       with (a) and (b). What happens to the capacity value of (b) as
       $p\to\infty$, and why does the version with the largest row norm stay
       above $1$?

1. [code] **Leaderboard attack.** ●● A leaderboard reports the 0-1 loss of
   each submitted label vector on a holdout set of $n=4{,}000$ examples with
   labels $\mathbf y\in\{0,1\}^n$. The *boosting attack* of
   @Blum.Hardt.2015 uses no knowledge of the task: submit $k$
   uniformly random label vectors, keep those with reported loss at most
   $\tfrac12$, and submit their coordinatewise majority vote. On fresh data,
   the majority vote has loss $\tfrac12$.
    1. Implement the attack for uniformly random $\mathbf y$ and
       $k\in\{10,30,100,300,1000,3000\}$, and record the reported loss of
       the majority vote. Compare its distance from $\tfrac12$ with the
       deviation that the equation allows for a
       pre-registered pool of $k+1$ models at $\delta=0.05$. From which $k$
       on does the attack exceed this deviation, and which hypothesis of the
       proposition does it violate?
    1. @Blum.Hardt.2015 prove that the reported loss is
       $\tfrac12-\Omega(\sqrt{k/n})$ but do not give the constant. Derive it
       heuristically with Gaussian approximations: estimate the expected
       excess agreement with $\mathbf y$ of a kept vector, then the
       probability that the majority of about $k/2$ such vectors is correct
       on one coordinate. Compare the constant with the measurements
       of (a).
    1. The *Ladder* mechanism of @Blum.Hardt.2015 releases a new
       score only when a submission beats the best released score by more
       than a step $\eta$, rounds it to a multiple of $\eta$, and otherwise
       repeats the previous score. Against the Ladder, the attacker can only
       keep the vectors that lowered the released score. Implement both for
       $\eta\in\{0.005,0.01\}$ and $k\in\{100,1000,3000\}$. How far below
       $\tfrac12$ does the majority vote now get, and how does this compare
       with the deviation of (a)?

1. **Ranking configurations on a holdout.** ● A validation set has
   $n=10{,}000$ examples, the loss takes values in $[0,1]$, and
   $\delta=0.05$.
    1. From the equation, how many configurations, fixed in
       advance, can be ranked with a simultaneous guarantee of $\pm2\%$,
       $\pm2.5\%$ and $\pm3\%$? Why is the count so sensitive to the
       tolerance?
    1. Repeat (a) with Chebyshev's inequality (variance at most $\tfrac14$)
       in place of Hoeffding's, with the same union bound. How many
       configurations does it allow at $\pm2\%$, and how many at $\pm5\%$?
    1. *The winner's curse.* Let all $k$ configurations have true error
       $10\%$, and treat their validation errors as independent. With the
       Gaussian approximation to each error and the estimate
       $\mathbb{E}[\max]\approx\sqrt{2\log k}$ for the maximum of $k$
       independent standard normals, show that the best reported error is
       about $10\%-0.3\%\cdot\sqrt{2\log k}$, and evaluate it at $k=1{,}000$.
       Then compute the exact median and the 5th and 95th percentiles of the
       minimum of $k=1{,}000$ independent $\textrm{Binomial}(10{,}000,0.1)$
       counts, divided by $n$, from the distribution function of the
       binomial.
    1. The reported score of the chosen configuration is biased downward by
       about one point. Explain in one sentence why scoring it on a fresh
       test set removes the bias. Does the bias violate a hypothesis of
       the equation, and does the bound cover it?

1. **Thresholds on the line.** ● Let
   $\mathcal F=\{x\mapsto\mathbf 1\{x\ge\theta\}:\theta\in\mathbb{R}\}$ act on
   $n$ distinct real points $x_1<\cdots<x_n$.
    1. Show that $\mathcal F$ realizes exactly $n+1$ labelings of the points
       and that its VC dimension (the largest number of points on which it
       realizes all labelings) is $1$. Show that with both orientations,
       $\mathbf 1\{x\ge\theta\}$ and $\mathbf 1\{x\le\theta\}$, the count is
       $2n$ and the VC dimension is $2$.
    1. Compute $\widehat{\mathfrak R}_S(\mathcal F)$ for the one-sided class
       exactly for $n=2$ and $n=3$ by enumerating the sign patterns.
    1. Show that in general
       $\widehat{\mathfrak R}_S(\mathcal F)=\frac1n\mathbb{E}\bigl[\max(0,\max_kT_k)\bigr]$,
       where $T_k$ is the sum of the signs at the $k$ largest points, a
       simple random walk. Argue with the reflection principle, or simulate,
       that this is about $\sqrt{2/(\pi n)}$ for large $n$, and check the
       values at $n=100$ and $n=1{,}000$.
    1. Compare with Massart's lemma applied to the $n+1$ labelings. By what
       factor is it loose at $n=100$, and which property of the labelings
       does it ignore?

1. [code] **The square case.** ● Use the sweep of the double-descent
   experiment in that section.
    1. Show from the stored norms that the nesting monotonicity fails below
       the threshold, and explain why the padding argument does not apply
       there.
    1. At $p=n$, decompose $\mathbf w$ along the equation.
       Report, as a median over trials, the coefficient
       $|\langle\mathbf u_n,\mathbf y\rangle|/\sigma_{\min}$ of the smallest
       singular direction next to $\|\mathbf w\|$. Which share of the norm
       does this one direction carry?
    1. Rerun the sweep with `noise = 0`. Then replace the teacher by a target
       $y=\langle\mathbf c,\boldsymbol\phi_{1:10}(\mathbf x)\rangle$ that is an
       exact combination of the first ten features, with $\|\mathbf c\|=1$
       and no noise. Describe the two curves. In each case, what does
       $\sigma_{\min}$ divide? Why does the minimum-norm interpolant become
       *worse* as features are added when the target uses only ten of them?
    1. Add label noise of standard deviation $0.1$ to the target of (c).
       Where is the test error now highest, how does the peak compare with
       the noise variance, and where is the error lowest?

1. [extended] **Benign spectra.** ●●
   @Bartlett.Long.Lugosi.Tsigler.2020 characterize when the minimum-norm interpolant in linear regression has
   small excess risk, the mechanism that
   that section describes in words. Let
   $\mathbf x\sim\mathcal N(\mathbf 0,\boldsymbol\Sigma)$ with
   $\boldsymbol\Sigma=\operatorname{diag}(\lambda_1,\ldots,\lambda_p)$ and
   $\lambda_1\ge\lambda_2\ge\cdots$, let
   $y=\langle\boldsymbol\theta^*,\mathbf x\rangle+\xi$ with
   $\xi\sim\mathcal N(0,\sigma^2)$, stack $n<p$ training inputs into
   $\mathbf X\in\mathbb R^{n\times p}$, and let
   $\hat{\boldsymbol\theta}=\mathbf X^+\mathbf y$. The excess risk is
   $(\hat{\boldsymbol\theta}-\boldsymbol\theta^*)^\top\boldsymbol\Sigma
   (\hat{\boldsymbol\theta}-\boldsymbol\theta^*)$.
    1. Show that its expectation over the noise is the sum of a bias term
       $\boldsymbol\theta^{*\top}(\mathbf I-\mathbf P)\boldsymbol\Sigma
       (\mathbf I-\mathbf P)\boldsymbol\theta^*$, where
       $\mathbf P=\mathbf X^+\mathbf X$, and a variance term
       $\sigma^2\operatorname{tr}\bigl((\mathbf X^+)^\top\boldsymbol\Sigma
       \mathbf X^+\bigr)$. Implement both terms; no test set is needed.
    1. The paper's effective ranks are
       $\rho_k=\sum_{i>k}\lambda_i/\lambda_{k+1}$ and
       $\varrho_k=\bigl(\sum_{i>k}\lambda_i\bigr)^2/\sum_{i>k}\lambda_i^2$. With
       $k^*$ the smallest $k$ such that $\rho_k\ge bn$, the variance term is of
       order $\sigma^2(k^*/n+n/\varrho_{k^*})$ up to constants; take $b=1$. Use
       $n=100$, $\sigma=1$, five strong directions with $\lambda_i=1$ that
       contain a unit-norm $\boldsymbol\theta^*$, and $q$ weak directions
       with $\lambda_i=1/q$ each, for $q\in\{100,300,10^3,3\cdot10^3,10^4\}$.
       Compare the measured variance with $\sigma^2(k^*/n+n/\varrho_{k^*})$. What
       happens at $q=100$, and why?
    1. Repeat with power-law spectra $\lambda_i=i^{-\alpha}$ truncated at
       $p=40n$, for $\alpha\in\{0.5,1,2\}$, $\boldsymbol\theta^*=\mathbf e_1$,
       and $n\in\{50,100,200,400\}$. For which spectra does the excess risk
       decrease with $n$? For the others, does the bias or the variance
       fail, and how do $\rho_0/n$, $k^*/n$, and $n/\varrho_{k^*}$ show it?

1. **Norms, not widths: a two-layer network.** ●● Let
   $\mathcal F_1=\{\mathbf x\mapsto\mathrm{ReLU}(\langle\mathbf v,\mathbf x\rangle):\|\mathbf v\|_2\le B_2\}$
   and
   $\mathcal F_2=\{\mathbf x\mapsto\sum_{j=1}^hw_j\,\mathrm{ReLU}(\langle\mathbf v_j,\mathbf x\rangle):\|\mathbf w\|_1\le B_1,\ \|\mathbf v_j\|_2\le B_2\}$,
   on points with $\|\mathbf x_i\|\le r$.
    1. Show that $\widehat{\mathfrak R}_S(\mathcal F_1)\le B_2r/\sqrt n$.
    1. Show that $\mathcal F_2$ is contained in $B_1$ times the convex hull of
       $\mathcal F_1\cup(-\mathcal F_1)$, and that taking the convex hull of a
       class does not change its empirical Rademacher complexity. Hint:
       $\sum_jw_jf_j=\|\mathbf w\|_1\sum_j\frac{|w_j|}{\|\mathbf w\|_1}(\pm f_j)$.
    1. Using $0\in\mathcal F_1$, show that
       $\mathbb{E}_{\boldsymbol\varepsilon}\sup_{f\in\mathcal F_1}|\sum_i\varepsilon_if(\mathbf x_i)|\le2n\,\widehat{\mathfrak R}_S(\mathcal F_1)$,
       and conclude that $\widehat{\mathfrak R}_S(\mathcal F_2)\le2B_1B_2r/\sqrt n$.
    1. The width $h$ does not appear in the bound. What does the bound say
       about weight decay on both layers? Why does it not explain the
       random-label experiment of that section?

[Discussions](https://d2l.discourse.group/t/36863)